# 02 — Model Training
Fine-tune EfficientNet-B4 on the fabric defect dataset and track experiments with MLflow.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import yaml
import torch
import mlflow
import matplotlib.pyplot as plt

CONFIG_PATH = '../configs/train_config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('Config loaded:')
print(yaml.dump(cfg, default_flow_style=False))

## Verify GPU Availability

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Start Training
Calls the training script from `src/detection/train.py`. Logs metrics to MLflow.

In [ ]:
from src.detection.train import main as train_main
train_main(CONFIG_PATH)

## View MLflow Results

In [ ]:
import mlflow

experiment = mlflow.get_experiment_by_name(cfg['output']['mlflow_experiment'])
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=['metrics.val_acc DESC'])
    print(runs[['run_id', 'metrics.val_acc', 'metrics.val_loss', 'metrics.train_acc']].head())

## Plot Training Curves

In [ ]:
if experiment and not runs.empty:
    run_id = runs.iloc[0]['run_id']
    history = mlflow.get_run(run_id)
    client = mlflow.tracking.MlflowClient()
    train_acc = [(m.step, m.value) for m in client.get_metric_history(run_id, 'train_acc')]
    val_acc = [(m.step, m.value) for m in client.get_metric_history(run_id, 'val_acc')]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(*zip(*train_acc), label='Train Acc')
    ax.plot(*zip(*val_acc), label='Val Acc')
    ax.axhline(0.92, color='red', linestyle='--', label='Target 92%')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_title('Training Curves'); ax.legend()
    plt.tight_layout(); plt.show()